# Pengujian Manual Pengenalan Wajah (Colab)
Notebook ini dibuat khusus agar Anda dapat mengunggah gambar langsung di Colab dan melihat hasil prediksi wajahnya secara interaktif.

### 1. Hubungkan Google Drive & Atur Path
Pastikan model YOLO dan Classifier (`.pkl`) Anda sudah tersimpan di Google Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Sesuaikan BASE_DIR dengan lokasi folder skripsi Anda di Google Drive
BASE_DIR = "/content/drive/MyDrive/skripsi"

YOLO_MODEL_PATH = os.path.join(BASE_DIR, "yolov10n-face.pt") # Atau ganti dengan 'medium.pt'
CLASSIFIER_SAVE_PATH = os.path.join(BASE_DIR, "face_classifier_model.pkl")
LABEL_ENCODER_SAVE_PATH = os.path.join(BASE_DIR, "label_encoder.pkl")

print("[INFO] Path telah diatur.")

### 2. Memuat Semua Model (YOLO, FaceNet, SVM)

In [ ]:
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from facenet_pytorch import InceptionResnetV1
from torchvision import transforms
import joblib
from google.colab.patches import cv2_imshow # Modul khusus Colab untuk menampilkan gambar

print("[INFO] Memuat Model...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load YOLO
try:
    yolo_model = YOLO(YOLO_MODEL_PATH)
except Exception as e:
    print(f"[ERROR] YOLO Gagal: {e}")

# Load FaceNet
facenet_model = InceptionResnetV1(pretrained='vggface2').eval().to(device)
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load Classifier SVM & Label Encoder
try:
    svm_classifier = joblib.load(CLASSIFIER_SAVE_PATH)
    label_encoder = joblib.load(LABEL_ENCODER_SAVE_PATH)
except Exception as e:
    print(f"[ERROR] Classifier Gagal: {e}")

print("[INFO] Semua Model Berhasil Dimuat!")

### 3. Upload Gambar & Lakukan Prediksi
Jalankan cell di bawah ini. Anda akan diminta untuk mengunggah (upload) sebuah foto. Hasilnya akan langsung digambar di layar.

In [ ]:
from google.colab import files

print("=== SILAKAN UPLOAD FOTO UNTUK DIUJI ===")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n[INFO] Memproses gambar: {filename}")
    img = cv2.imread(filename)
    
    if img is None:
        print("[ERROR] Gambar gagal dibaca.")
        continue
        
    img_draw = img.copy()
    results = yolo_model(img, verbose=False)
    wajah_terdeteksi = 0
    
    if len(results) > 0 and len(results[0].boxes) > 0:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        
        for box in boxes:
            wajah_terdeteksi += 1
            x1, y1, x2, y2 = map(int, box)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)
            
            face_crop = img[y1:y2, x1:x2]
            if face_crop.size == 0: continue
                
            # Ekstraksi dengan FaceNet
            face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
            face_tensor = transform(face_rgb).unsqueeze(0).to(device)
            with torch.no_grad():
                embedding = facenet_model(face_tensor).cpu().numpy()
                
            # Prediksi dengan SVM
            probabilities = svm_classifier.predict_proba(embedding)[0]
            max_prob_index = np.argmax(probabilities)
            max_prob = probabilities[max_prob_index]
            
            # Mengambil Nama dari Label Encoder
            predicted_name = label_encoder.inverse_transform([max_prob_index])[0]
            
            # Jika Confidence Score di bawah 50%, anggap Unknown
            if max_prob < 0.50:
                predicted_name = "Unknown"
                
            teks_label = f"{predicted_name} ({max_prob*100:.1f}%)"
            
            # Menggambar kotak dan teks
            color = (0, 255, 0) if predicted_name != "Unknown" else (0, 0, 255)
            cv2.rectangle(img_draw, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img_draw, teks_label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            
        print(f"[SUCCESS] Berhasil mendeteksi {wajah_terdeteksi} wajah.")
        # Tampilkan gambar langsung di antarmuka Colab
        cv2_imshow(img_draw) 
    else:
        print("\n[WARNING] Tidak ada wajah yang terdeteksi pada gambar ini.")